# MoE Routing Convergence — Analysis Notebook

This notebook applies the Fused Gromov-Wasserstein (FGW) metric (see `experiments/fgw.py`
and `6a154a47401c9f4881c67a3f/main.tex`) to compare routing DAGs across MoE models and tasks.

**Structure**
- §0 Setup
- §1 Routing-graph visualisation

## §0. Setup

In [1]:
from __future__ import annotations
from pathlib import Path 

import yaml, json, os, sys

import numpy as np, matplotlib.pyplot as plt
import igraph as ig, networkx as nx
import torch

from collections import deque 
from matplotlib.ticker import MultipleLocator, FuncFormatter
import matplotlib.colors as mcolors


ROOT = "/scratch/sleonard/MoE_circuits"
sys.path.insert(0, ROOT)

with open(os.path.join(ROOT, "config.yaml")) as f:
    config = yaml.safe_load(f)

# DAG schema (post-rebuild; cf. experiments/build_dag.py header):
#   Edge tensors [c, j, l, n]:
#     APS, ANS                     positive / negative parts of the per-edge sub-score
#     AARV                         mean |rank shift| under sender ablation
#     P_add, P_rem                 prob. receiver crosses INTO / OUT OF top-K
#     W_softmax (PRIMARY)          E_i[|p_orig(v) - p_pert(v)|] - softmax-mass perturbation
#     W_softmax_var                Var_i[|p_orig - p_pert|]
#     W_softmax_signed             E_i[p_pert - p_orig]  (>0 means sender suppresses receiver)
#   Per-vertex:
#     act [L, N]                   Su et al. activation magnitude (max_i ||down_proj||_inf)
#     n_tokens_selected [L, N]
#     top_weight / top_prompt / top_pos / top_token  [L, N, k_top_tokens]
#   Scalars / metadata:
#     k_top_tokens, n_prompts, max_tokens, model, moe_layers, dataset

# MODELS   = ["deepseek-v2", "deepseek-v2-lite", "mixtral-8x7b", "mixtral-8x22b", "qwen3-30b-a3b", "qwen3-235b-a22b", "olmoe", "phi-3.5-moe"]
MODELS = ["deepseek-v2", "deepseek-v2-lite", "mixtral-8x7b", "mixtral-8x22b", "qwen3-30b-a3b", "qwen3-235b-a22b", "olmoe", "phi-3.5-moe"]
DATASETS = ["c4"]

dags: dict[tuple[str, str], dict] = {}
for m in MODELS:
    for d in DATASETS:
        path = os.path.join(config["result_path"], f"circuits/dag_{m}_{d}.pt")
        # weights_only=False because the dag dict contains plain-Python entries
        # (model string, moe_layers list, dataset name).
        dags[(m, d)] = torch.load(path, map_location="cpu", weights_only=False)


/scratch/sleonard/miniconda3/envs/megatron/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## §1. Routing-graph visualisation

Per-model vertex-first and edge-first sparsifications, rendered as layered graphs.
*Depends on §0.* Independent of the FGW pipeline below.

In [ ]:
from experiments.helper import (
    sparsify_edges, thresholding_routing_graph, show_enhanced_layered_graph,
)

# Edge-first visualisation of the routing DAG on W_softmax / c4.
#
#   W_softmax(v -> v') = E_i[|p_orig(v') - p_pert(v')|] under an ablation of v,
#   as defined in main.tex; bounded in [0, 1].
#
# Sparsification: keep edges with |W_softmax| above the EDGE_Q quantile of
# all forward-edge magnitudes (top (1 - EDGE_Q) fraction). No per-vertex
# threshold, no vertex-first pass.
#
# Vertex handling: show_enhanced_layered_graph already drops isolated
# vertices (degree == 0) from the drawing, so no extra work is needed here.
# We do NOT set an `is_super` vertex attribute, which suppresses the
# gold/red highlighting inside the drawing routine — every node renders
# uniformly (white fill, black border).
TARGET = "W_softmax"
DATASET = "c4"
EDGE_Q = 0.999

# Colour axis fixed at [0, 1] so the same |w| renders the same shade in
# every model.

for m in MODELS:
    dag = dags[(m, DATASET)]
    W = dag[TARGET]
    L, N = W.shape[0], W.shape[1]
    layer_labels = dag["moe_layers"]

    W_e, einfo = sparsify_edges(W, edge_q=EDGE_Q, edge_floor_frac=0.0)
    print(f"[EDGE] {m}: edges_kept={einfo['n_edges_kept']}/{einfo['n_edges_total']}, "
          f"t_edge={einfo['t_edge']:.4g}")

    dag["_vis_edge"] = W_e
    g_e = thresholding_routing_graph(dag, "_vis_edge", 1e-9)

    show_enhanced_layered_graph(
        g_e, quantile=EDGE_Q,
        target=f"{TARGET}/EDGE-first (edge_q={EDGE_Q}, n_kept={einfo['n_edges_kept']})",
        model=m, dataset=DATASET, n_prompts=dag["n_prompts"],
        layer_labels=layer_labels,
        color_vmin=0.0, color_vmax=1.0,
    )


[EDGE] deepseek-v2: edges_kept=43261/43260203, t_edge=0.04126
